# 10 — Adaptive QEM Hardware Campaign

This notebook is the **real experimental campaign controller**. It consumes the calibration snapshot and adaptive selection plan produced by Notebook 09/07, then prepares the raw, readout-mitigation, and ZNE hardware experiments.

**Default:** `SUBMIT_JOBS = False`. Nothing is submitted until the environment and experiment plan have been reviewed.

IBM's current SamplerV2 interface accepts a list of PUBs/circuits and a `shots` argument. citeturn0search1turn0search0


In [ ]:
from pathlib import Path
import sys, json, time
import pandas as pd

ROOT = Path.cwd()
if (ROOT / "execute_ibm_qem.py").exists():
    pass
elif (ROOT / "Adaptive_QEM_IBM").exists():
    ROOT = ROOT / "Adaptive_QEM_IBM"

sys.path.insert(0, str(ROOT))

from execute_ibm_qem import load_service, get_backend, prepare_circuit, submit_sampler_v2
from qiskit_ibm_runtime import SamplerV2

BACKEND_NAME = "ibm_kingston"
SHOTS = 4096
OPTIMIZATION_LEVEL = 3
SEED_TRANSPIILER = 42
SUBMIT_JOBS = False

print("ROOT:", ROOT)
print("Backend:", BACKEND_NAME)
print("Shots:", SHOTS)
print("SUBMIT_JOBS:", SUBMIT_JOBS)


## 1. Load the adaptive execution matrix

In [ ]:
plan_file = ROOT / "data/hardware/adaptive_qem_execution_matrix.csv"
selection_file = ROOT / "data/hardware/adaptive_qem_selection_plan.csv"

if not plan_file.exists():
    raise FileNotFoundError(
        "Run Notebook 07 first to create adaptive_qem_execution_matrix.csv."
    )

execution_df = pd.read_csv(plan_file)
selection_df = pd.read_csv(selection_file)

display(selection_df)
display(execution_df)
print("Planned execution rows:", len(execution_df))


## 2. Hardware authentication and calibration provenance

The campaign should use the same backend identified by the calibration snapshot. Calibration is time-dependent, so the timestamp/snapshot must remain attached to every experimental batch.


In [ ]:
service = load_service()
backend = get_backend(service, BACKEND_NAME)

print("Backend:", backend.name)
print("Operational:", backend.status().operational)
print("Pending jobs:", backend.status().pending_jobs)
print("Qubits:", backend.num_qubits)

cal_manifest = ROOT / "data/calibration/ibm_kingston_calibration_manifest.json"
if cal_manifest.exists():
    calibration = json.loads(cal_manifest.read_text(encoding="utf-8"))
    print("Calibration manifest loaded.")
    print(json.dumps(calibration, indent=2))
else:
    print("WARNING: calibration manifest not found.")


## 3. Load benchmark circuits

In [ ]:
from circuits.bell import bell_phi_plus
from circuits.ghz import create_ghz
from circuits.teleportation import teleportation
from circuits.superdense import superdense_coding
from circuits.qft import qft
from circuits.grover import grover_2qubit
from circuits.qaoa import qaoa_two_node
from circuits.qpe import qpe
from circuits.qrng import qrng

benchmarks = {
    "Bell_Phi_Plus": bell_phi_plus(),
    "GHZ_3": create_ghz(3),
    "GHZ_4": create_ghz(4),
    "GHZ_5": create_ghz(5),
    "Teleportation": teleportation(),
    "Superdense_00": superdense_coding("00"),
    "Superdense_01": superdense_coding("01"),
    "Superdense_10": superdense_coding("10"),
    "Superdense_11": superdense_coding("11"),
    "QFT_3": qft(3),
    "QFT_4": qft(4),
    "Grover_2Q": grover_2qubit(),
    "QAOA_2Q": qaoa_two_node(),
    "QPE": qpe(),
    "QRNG_4": qrng(4),
}
print("Loaded", len(benchmarks), "benchmarks.")


## 4. Build the raw hardware baseline

Every benchmark receives a raw baseline. This is essential because adaptive QEM must be evaluated against the same unmitigated hardware result.


In [ ]:
prepared = {}
for name, circuit in benchmarks.items():
    prepared[name] = prepare_circuit(
        circuit,
        backend,
        optimization_level=OPTIMIZATION_LEVEL,
        seed_transpiler=SEED_TRANSPIILER,
    )

characterization = []
for name, c in prepared.items():
    characterization.append({
        "circuit": name,
        "qubits": c.num_qubits,
        "clbits": c.num_clbits,
        "depth": c.depth(),
        "size": c.size(),
        "cx_count": c.count_ops().get("cx", 0),
        "cz_count": c.count_ops().get("cz", 0),
        "swap_count": c.count_ops().get("swap", 0),
    })

char_df = pd.DataFrame(characterization)
display(char_df)


## 5. Create hardware job batches

To preserve provenance, each execution group is explicitly labeled. For ZNE, scale factors 1/3/5 are separate physical executions of folded circuits. For readout mitigation, the raw circuit is retained and calibration correction is performed offline using the calibration matrix.


In [ ]:
batches = []

for _, row in selection_df.iterrows():
    name = row["circuit"]
    method = row["method"]

    batches.append({
        "circuit": name,
        "strategy": "raw",
        "scale_factor": 1,
    })

    if method in ("readout_mitigation", "combined"):
        batches.append({
            "circuit": name,
            "strategy": "readout_mitigation",
            "scale_factor": 1,
        })

    if method in ("zne", "combined"):
        for sf in (1, 3, 5):
            batches.append({
                "circuit": name,
                "strategy": "zne",
                "scale_factor": sf,
            })

batch_df = pd.DataFrame(batches)
batch_df = batch_df.drop_duplicates()
display(batch_df)
print("Physical experiment rows:", len(batch_df))


## 6. Dry-run validation

Before submission, check that every circuit exists, every circuit has measurements, and the estimated shot volume is acceptable.


In [ ]:
missing = [n for n in batch_df["circuit"] if n not in prepared]
if missing:
    raise ValueError(f"Missing prepared circuits: {missing}")

if any(prepared[n].num_clbits == 0 for n in batch_df["circuit"]):
    raise ValueError("At least one circuit has no classical measurement register.")

print("Validation passed.")
print("Total requested circuit-shots:", len(batch_df) * SHOTS)


## 7. Submit raw baseline batch

Set `SUBMIT_JOBS=True` only after reviewing the dry-run output.

SamplerV2 can submit multiple circuits in one call, with `shots` applying per PUB when no PUB-specific shot count is supplied. citeturn0search0turn0search1


In [ ]:
job_records = []

if SUBMIT_JOBS:
    raw_names = list(dict.fromkeys(batch_df.loc[batch_df["strategy"] == "raw", "circuit"]))
    raw_circuits = [prepared[n] for n in raw_names]

    job = submit_sampler_v2(backend, raw_circuits, shots=SHOTS)

    record = {
        "backend": backend.name,
        "job_id": job.job_id(),
        "strategy": "raw",
        "shots": SHOTS,
        "circuits": raw_names,
        "submitted_at_unix": time.time(),
        "optimization_level": OPTIMIZATION_LEVEL,
        "seed_transpiler": SEED_TRANSPIILER,
    }

    job_records.append(record)

    out = ROOT / "data/hardware/jobs"
    out.mkdir(parents=True, exist_ok=True)
    (out / f"raw_{job.job_id()}.json").write_text(
        json.dumps(record, indent=2), encoding="utf-8"
    )

    print("RAW JOB ID:", job.job_id())
else:
    print("DRY RUN — raw job not submitted.")


## 8. ZNE batch preparation

ZNE is performed using circuit folding. Scale factor 1 is the original circuit; 3 and 5 increase the physical gate sequence while preserving the ideal unitary for unitary operations.

The ZNE implementation should exclude measurement/barrier operations from folding.


In [ ]:
from mitigation.zne import fold_circuit

zne_circuits = {}
for _, row in batch_df[batch_df["strategy"] == "zne"].iterrows():
    name = row["circuit"]
    sf = int(row["scale_factor"])
    key = f"{name}__sf{sf}"
    zne_circuits[key] = fold_circuit(prepared[name], sf)

print("Prepared ZNE circuits:", len(zne_circuits))
for key, c in list(zne_circuits.items())[:10]:
    print(key, "depth=", c.depth(), "size=", c.size())


## 9. Submit ZNE jobs

For rigorous overhead accounting, record each scale-factor job ID separately. Do not combine ZNE results with raw results without preserving the scale factor.


In [ ]:
if SUBMIT_JOBS and zne_circuits:
    zne_names = list(zne_circuits.keys())
    zne_list = [zne_circuits[n] for n in zne_names]

    job = submit_sampler_v2(backend, zne_list, shots=SHOTS)

    record = {
        "backend": backend.name,
        "job_id": job.job_id(),
        "strategy": "zne",
        "shots": SHOTS,
        "scale_factors": [1, 3, 5],
        "circuits": zne_names,
        "submitted_at_unix": time.time(),
        "optimization_level": OPTIMIZATION_LEVEL,
    }

    job_records.append(record)

    out = ROOT / "data/hardware/jobs"
    out.mkdir(parents=True, exist_ok=True)
    (out / f"zne_{job.job_id()}.json").write_text(
        json.dumps(record, indent=2), encoding="utf-8"
    )

    print("ZNE JOB ID:", job.job_id())
else:
    print("DRY RUN — ZNE job not submitted.")


## 10. Save campaign manifest

This file is the provenance anchor for the paper. It records the backend, software-independent experiment configuration, adaptive selections, and job records.


In [ ]:
out = ROOT / "data/hardware"
out.mkdir(parents=True, exist_ok=True)

manifest = {
    "backend": BACKEND_NAME,
    "shots": SHOTS,
    "optimization_level": OPTIMIZATION_LEVEL,
    "seed_transpiler": SEED_TRANSPIILER,
    "submit_jobs": SUBMIT_JOBS,
    "adaptive_selection_file": str(selection_file),
    "execution_matrix_file": str(plan_file),
    "characterization_file": "hardware_campaign_characterization.csv",
    "job_records": job_records,
}

char_df.to_csv(out / "hardware_campaign_characterization.csv", index=False)
(out / "adaptive_qem_hardware_campaign_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

print("Campaign manifest saved to:", out.resolve())


## Experimental protocol

For the paper, the campaign should report:

- backend and calibration snapshot/time;
- Qiskit and Runtime versions;
- optimization level and transpiler seed;
- shots per circuit;
- raw job IDs;
- mitigation job IDs;
- ZNE scale factors;
- raw counts and mitigated distributions;
- success probability;
- distribution fidelity;
- TVD;
- execution count / mitigation overhead;
- uncertainty intervals from repeated runs where feasible.

**Do not call the adaptive selector validated until the resulting hardware data demonstrate its behavior against fixed baselines.**
